# Notebook 07: Model Evaluation & Selection

## Step 1: Environment Setup & Spark Session

In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import time

spark = SparkSession.builder \
    .appName("SmartCityBusClustering_Evaluation") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/25 08:58:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/25 08:58:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/25 08:58:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 4.1.1


## Step 2: Load Feature Dataset & Rebuild Best Models

In [2]:
PROJECT_ROOT = "/Users/aayushbohara/Desktop/smartcity-bus-clustering"
FEATURES_PATH = os.path.join(PROJECT_ROOT, "data/processed/features_dataset.parquet")

df = spark.read.parquet(FEATURES_PATH)

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.sql.functions import coalesce, lit

CLEANED_PATH = os.path.join(PROJECT_ROOT, "data/processed/cleaned_dataset.csv")
direction_lookup = spark.read.csv(CLEANED_PATH, header=True, inferSchema=True) \
    .select("timestamp", "vehicleRef", "lineRef", "directionRef") \
    .dropDuplicates(["timestamp", "vehicleRef", "lineRef"])

df = df.join(direction_lookup, on=["timestamp", "vehicleRef", "lineRef"], how="left")
df = df.withColumn("directionRef", coalesce(col("directionRef"), lit("unknown"))) \
       .withColumn("nationalOperatorCode", coalesce(col("nationalOperatorCode"), lit("unknown")))

direction_indexer = StringIndexer(inputCol="directionRef", outputCol="directionRef_idx", handleInvalid="keep")
operator_indexer = StringIndexer(inputCol="nationalOperatorCode", outputCol="operator_idx", handleInvalid="keep")
direction_encoder = OneHotEncoder(inputCol="directionRef_idx", outputCol="directionRef_ohe")
operator_encoder = OneHotEncoder(inputCol="operator_idx", outputCol="operator_ohe")

numeric_cols = ["latitude", "longitude", "hour_of_day",
                "nearest_schedule_deviation_minutes_v2", "speed_kmh_capped", "disruption_count"]

assembler = VectorAssembler(inputCols=numeric_cols + ["directionRef_ohe", "operator_ohe"], outputCol="features_raw_v2")
scaler = StandardScaler(inputCol="features_raw_v2", outputCol="features_v2", withMean=True, withStd=True)

full_pipeline = Pipeline(stages=[direction_indexer, operator_indexer, direction_encoder, operator_encoder, assembler, scaler])
full_pipeline_model = full_pipeline.fit(df)
df_features_v2 = full_pipeline_model.transform(df)

train_full, test_full = df_features_v2.randomSplit([0.8, 0.2], seed=42)
train_full.cache()
test_full.cache()

print("Train rows:", train_full.count())
print("Test rows:", test_full.count())

Train rows: 617500
Test rows: 154233


## Step 3: Retrain Final Models at Best Hyperparameters

In [3]:
from pyspark.ml.clustering import KMeans, GaussianMixture
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features_v2", predictionCol="cluster",
    metricName="silhouette", distanceMeasure="squaredEuclidean"
)

# --- K-Means (k=6) ---
start = time.time()
kmeans_final = KMeans(featuresCol="features_v2", predictionCol="cluster", k=6, seed=42).fit(train_full)
kmeans_time = time.time() - start

kmeans_preds = kmeans_final.transform(test_full)
kmeans_silhouette = evaluator.evaluate(kmeans_preds)
kmeans_wcss = kmeans_final.summary.trainingCost

print(f"K-Means: time={kmeans_time:.1f}s, silhouette={kmeans_silhouette:.4f}, wcss={kmeans_wcss:.1f}")

# --- GMM (k=5) ---
start = time.time()
gmm_final = GaussianMixture(featuresCol="features_v2", predictionCol="cluster", k=5, seed=42).fit(train_full)
gmm_time = time.time() - start

gmm_preds = gmm_final.transform(test_full)
gmm_silhouette = evaluator.evaluate(gmm_preds)

print(f"GMM: time={gmm_time:.1f}s, silhouette={gmm_silhouette:.4f}")

26/07/25 09:00:01 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/07/25 09:00:03 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


K-Means: time=3.2s, silhouette=0.4558, wcss=4241739.7


26/07/25 09:00:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


GMM: time=15.4s, silhouette=0.2740


## Step 4: Final Comparison Table — All 3 Models

In [4]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "K-Means (baseline)",
        "Best Params": "k=6",
        "Silhouette Score": round(kmeans_silhouette, 4),
        "WCSS": round(kmeans_wcss, 1),
        "Training Time (s)": round(kmeans_time, 1),
        "Time Complexity": "O(n·k·i·d)",
        "Verdict": "SELECTED - balanced, interpretable clusters"
    },
    {
        "Model": "Gaussian Mixture Model",
        "Best Params": "k=5",
        "Silhouette Score": round(gmm_silhouette, 4),
        "WCSS": "N/A (probabilistic)",
        "Training Time (s)": round(gmm_time, 1),
        "Time Complexity": "O(n·k·d²·i)",
        "Verdict": "Rejected - Gaussian assumption violated by categorical features"
    },
    {
        "Model": "DBSCAN",
        "Best Params": "eps=0.8, min_samples=10 (meaningful config)",
        "Silhouette Score": 0.168,
        "WCSS": "N/A (density-based)",
        "Training Time (s)": 6.4,
        "Time Complexity": "O(n log n) with spatial index",
        "Verdict": "Rejected - degenerate at high eps, fragmented at low eps"
    },
])

print(comparison.to_string(index=False))

                 Model                                 Best Params  Silhouette Score                WCSS  Training Time (s)               Time Complexity                                                         Verdict
    K-Means (baseline)                                         k=6            0.4558           4241739.7                3.2                    O(n·k·i·d)                     SELECTED - balanced, interpretable clusters
Gaussian Mixture Model                                         k=5            0.2740 N/A (probabilistic)               15.4                   O(n·k·d²·i) Rejected - Gaussian assumption violated by categorical features
                DBSCAN eps=0.8, min_samples=10 (meaningful config)            0.1680 N/A (density-based)                6.4 O(n log n) with spatial index        Rejected - degenerate at high eps, fragmented at low eps


## Step 5: Algorithmic Complexity Analysis (B1)

| Model | Time Complexity | Space Complexity | Notes |
|---|---|---|---|
| K-Means | O(n·k·i·d) | O(n·d + k·d) | n=617,500, k=6, d=13, converges in few iterations (i) — cheapest model, confirmed by 3.2s wall-clock time |
| GMM | O(n·k·d²·i) | O(n·d + k·d²) | The d² term (covariance matrices) makes this ~5x slower than K-Means in practice (15.4s vs 3.2s) — theory matches measurement |
| DBSCAN | O(n log n) with spatial index, O(n²) worst case | O(n) | sklearn uses a KD-tree internally; measured 6.4s on 38,753-row sample (not full 617K rows, since DBSCAN requires driver-side memory) |

**Practical implication:** K-Means' linear-in-d complexity made it both the 
fastest and most accurate model on this feature space. GMM's quadratic 
dependency on dimensionality (d=13, including one-hot encoded categoricals) 
directly explains both its slower runtime and its accuracy penalty, since 
covariance estimation degrades when several dimensions are binary rather 
than continuous.

In [5]:
complexity_summary = pd.DataFrame([
    {"Model": "K-Means", "Time Complexity": "O(n·k·i·d)", "Space Complexity": "O(n·d + k·d)",
     "Measured Time (s)": 3.2, "n": 617500, "k": 6, "d": 13},
    {"Model": "GMM", "Time Complexity": "O(n·k·d²·i)", "Space Complexity": "O(n·d + k·d²)",
     "Measured Time (s)": 15.4, "n": 617500, "k": 5, "d": 13},
    {"Model": "DBSCAN", "Time Complexity": "O(n log n) [indexed] / O(n²) [naive]", "Space Complexity": "O(n)",
     "Measured Time (s)": 6.4, "n": 38753, "k": "N/A (density-based)", "d": 13},
])
print(complexity_summary.to_string(index=False))

  Model                      Time Complexity Space Complexity  Measured Time (s)      n                   k  d
K-Means                           O(n·k·i·d)     O(n·d + k·d)                3.2 617500                   6 13
    GMM                          O(n·k·d²·i)    O(n·d + k·d²)               15.4 617500                   5 13
 DBSCAN O(n log n) [indexed] / O(n²) [naive]             O(n)                6.4  38753 N/A (density-based) 13


## Step 6: Save Final Model (K-Means) for Dashboard Integration

In [6]:
MODEL_PATH = os.path.join(PROJECT_ROOT, "data/models/kmeans_final")
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

kmeans_final.write().overwrite().save(MODEL_PATH)
full_pipeline_model.write().overwrite().save(os.path.join(PROJECT_ROOT, "data/models/feature_pipeline"))

print("K-Means model saved to:", MODEL_PATH)
print("Feature pipeline saved to:", os.path.join(PROJECT_ROOT, "data/models/feature_pipeline"))

K-Means model saved to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/models/kmeans_final
Feature pipeline saved to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/models/feature_pipeline


## Step 7: Export Dashboard Dataset

Apply the final K-Means model to the full dataset and export a lightweight 
CSV (no vector columns) for the Streamlit dashboard, avoiding a Spark 
dependency inside the app itself.

In [7]:
full_predictions = kmeans_final.transform(df_features_v2)

dashboard_df = full_predictions.select(
    "timestamp", "lineRef", "vehicleRef", "nationalOperatorCode", "directionRef",
    "latitude", "longitude", "hour_of_day",
    "nearest_schedule_deviation_minutes_v2", "speed_kmh_capped", "disruption_count",
    "cluster"
)

DASHBOARD_DATA_PATH = os.path.join(PROJECT_ROOT, "data/processed/dashboard_data.csv")
dashboard_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(DASHBOARD_DATA_PATH)

import glob
part_file = glob.glob(os.path.join(DASHBOARD_DATA_PATH, "*.csv"))[0]
final_path = os.path.join(PROJECT_ROOT, "data/processed/dashboard_data_final.csv")
os.rename(part_file, final_path)

print("Dashboard dataset exported to:", final_path)

import pandas as pd
check = pd.read_csv(final_path)
print("Rows:", len(check))
print(check["cluster"].value_counts().sort_index())

Dashboard dataset exported to: /Users/aayushbohara/Desktop/smartcity-bus-clustering/data/processed/dashboard_data_final.csv
Rows: 771733
cluster
0    183581
1    226351
2     30111
3     42568
4     33648
5    255474
Name: count, dtype: int64
